# 1. Import the Data

In [113]:
import pandas as pd

df = pd.read_csv("data.csv")
df.head()

,title,score,comments
0,Removable batteries in smartphones will be man...,170,138
1,Redis array: short story of a long development...,70,16
2,GitHub Is Down,84,28
3,Talking to 35 Strangers at the Gym,562,274
4,How Monero's proof of work works,71,41


In [114]:
# Clean the text
df["title"] = df["title"].str.lower()

# Create labels
threshold = df["score"].quantile(0.7)           # We will consider the top 30% scored posts as positive, instead of using a fixed arbitrary threshold
df["label"] = (df["score"] > threshold).astype(int)

# Selecting only what we need
df = df[["title", "label"]]
df.head()

,title,label
0,removable batteries in smartphones will be man...,0
1,redis array: short story of a long development...,0
2,github is down,0
3,talking to 35 strangers at the gym,1
4,how monero's proof of work works,0


In [115]:
# Save new csv

df.to_csv("clean_data.csv", index=False)

# 2. Text Tokenization and Numerical Encoding

In [116]:
import torch
from collections import Counter

# Tokenization + Encoding

# Build vocabulary

all_words = " ".join(df["title"]).split()
word_counts = Counter(all_words)

vocab = {word: i+1 for i, (word, _) in enumerate(word_counts.most_common(10000))}


# Encode text

def encode(text):
    return [vocab.get(word, 0) for word in text.split()]

df["encoded"] = df["title"].apply(encode)


# Pad sequences

max_len = 10

def pad(seq):
    return seq[:max_len] + [0]*(max_len - len(seq))

df["padded"] = df["encoded"].apply(pad)

# Check
df.head()


,title,label,encoded,padded
0,removable batteries in smartphones will be man...,0,"[490, 491, 4, 492, 52, 46, 493, 4, 1, 494, 495...","[490, 491, 4, 492, 52, 46, 493, 4, 1, 494]"
1,redis array: short story of a long development...,0,"[497, 498, 187, 188, 6, 2, 189, 499, 500]","[497, 498, 187, 188, 6, 2, 189, 499, 500, 0]"
2,github is down,0,"[23, 8, 190]","[23, 8, 190, 0, 0, 0, 0, 0, 0, 0]"
3,talking to 35 strangers at the gym,1,"[191, 5, 501, 502, 24, 1, 503]","[191, 5, 501, 502, 24, 1, 503, 0, 0, 0]"
4,how monero's proof of work works,0,"[17, 504, 505, 6, 53, 192]","[17, 504, 505, 6, 53, 192, 0, 0, 0, 0]"


# 3. The Model

## Model Definition

In [117]:
# Convert to tensors

X = torch.tensor(df["padded"].tolist(), dtype=torch.long)
y = torch.tensor(df["label"].values, dtype=torch.float32)

print(X.shape, y.shape)

torch.Size([500, 10]) torch.Size([500])


In [118]:
df["label"].value_counts()

label
0    351
1    149
Name: count, dtype: int64

In [119]:
# Train test split
from sklearn.model_selection import train_test_split

# Stratify to preserve class distribution across train and test sets for fair evaluation and avoid a "lazy" model.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [130]:
# Define the model
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc = nn.Linear(embed_dim, 1)

    def forward(self, x):
        x = self.embedding(x)
        x = x.mean(dim=1)
        x = self.fc(x)
        return x   # ← raw logits (NO sigmoid)

In [131]:
# Initialize the model

vocab_size = len(vocab) + 1
model = SimpleModel(vocab_size, embed_dim=16)

In [132]:
# Training setup

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
pos_weight = torch.tensor(pos_weight, dtype=torch.float32)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

C:\Users\ajdpe\AppData\Local\Temp\ipykernel_12640\469144809.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pos_weight = torch.tensor(pos_weight, dtype=torch.float32)


In [133]:
# Training loop
for epoch in range(10):
    model.train()
    
    outputs = model(X_train).squeeze()
    loss = criterion(outputs, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.9840
Epoch 2, Loss: 0.9757
Epoch 3, Loss: 0.9675
Epoch 4, Loss: 0.9590
Epoch 5, Loss: 0.9500
Epoch 6, Loss: 0.9405
Epoch 7, Loss: 0.9305
Epoch 8, Loss: 0.9198
Epoch 9, Loss: 0.9086
Epoch 10, Loss: 0.8966


In [142]:
# Test predictions
model.eval()

with torch.no_grad():
    logits = model(X_test).squeeze()
    
    # Convert logits → probabilities
    preds = torch.sigmoid(logits)
    
    print("Probabilities:", preds[:10])
    
    threshold = 0.5
    pred_labels = (preds > threshold).float()
    
    accuracy = (pred_labels == y_test).float().mean()
    print("Test Accuracy:", accuracy.item())
    
    print("\nPredictions vs True:")
    for i in range(10):
        print(f"Prob: {preds[i]:.3f} | Pred: {pred_labels[i].item()} | True: {y_test[i].item()}")

Probabilities: tensor([0.4781, 0.5698, 0.4684, 0.5980, 0.5229, 0.5304, 0.5748, 0.4882, 0.4203,
        0.5789])
Test Accuracy: 0.47999998927116394

Predictions vs True:
Prob: 0.478 | Pred: 0.0 | True: 0.0
Prob: 0.570 | Pred: 1.0 | True: 0.0
Prob: 0.468 | Pred: 0.0 | True: 0.0
Prob: 0.598 | Pred: 1.0 | True: 0.0
Prob: 0.523 | Pred: 1.0 | True: 1.0
Prob: 0.530 | Pred: 1.0 | True: 0.0
Prob: 0.575 | Pred: 1.0 | True: 1.0
Prob: 0.488 | Pred: 0.0 | True: 0.0
Prob: 0.420 | Pred: 0.0 | True: 0.0
Prob: 0.579 | Pred: 1.0 | True: 0.0


In [ ]:
print(pred_labels)
print(y_test)

tensor([0., 1., 0., 1., 1., 1., 1., 0., 0., 1., 1., 1., 1., 0., 0., 1., 1., 1.,
        1., 0., 0., 1., 1., 0., 0., 0., 1., 1., 1., 1., 0., 0., 1., 1., 1., 0.,
        1., 0., 1., 1., 0., 0., 0., 1., 1., 1., 1., 0., 1., 1., 0., 1., 0., 1.,
        0., 1., 1., 1., 1., 0., 0., 1., 1., 1., 0., 1., 1., 0., 1., 1., 1., 1.,
        1., 0., 1., 0., 1., 1., 1., 0., 0., 1., 1., 0., 0., 1., 1., 0., 0., 1.,
        1., 1., 1., 1., 1., 1., 0., 1., 1., 0.])
tensor([0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 1., 0., 0., 0., 0., 1., 1., 0.,
        0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
        0., 1., 0., 0., 1., 1., 1., 0., 0., 1., 1., 0., 0., 0., 0., 1., 0., 1.,
        0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1.,
        0., 0., 1., 1., 0., 1., 0., 1., 1., 1., 1., 0., 0., 1., 0., 0., 0., 0.,
        0., 0., 0., 1., 1., 0., 1., 1., 0., 0.])
              precision    recall  f1-score   support

         0.0       0.75      0.39      0.51    

In [143]:
# Look at classification report
from sklearn.metrics import classification_report

print(classification_report(y_test, pred_labels))

              precision    recall  f1-score   support

         0.0       0.75      0.39      0.51        70
         1.0       0.33      0.70      0.45        30

    accuracy                           0.48       100
   macro avg       0.54      0.54      0.48       100
weighted avg       0.62      0.48      0.49       100



## Conclusion

- Introducing class-weighted loss significantly changed the model’s behavior.

- While overall accuracy decreased, the model became much better at identifying the minority class (high-engagement posts), increasing recall for class 1 from very low values to approximately 0.70.

- Adjusting the classification threshold revealed a clear trade-off between precision and recall. Lower thresholds increase recall for the minority class but introduce more false positives, while higher thresholds improve precision but significantly reduce the model’s ability to detect relevant cases.

- In this project, a threshold of 0.5 provides the best balance between identifying high-engagement posts and limiting incorrect predictions.

Started simple → saw bias → fixed imbalance → tuned threshold → analyzed trade-offs


In [ ]:
torch.save(model.state_dict(), "model.pt") # saving the weights of the model to then use in predict.py
torch.save(vocab, "vocab.pt")